# Modeling: Predicting Home Wins

Predict whether the **home team wins** using only information known before kickoff.

- **Data:** the 2004-2025 seasons. 2002-2003 are left out because they have no box-score stats (they're only used for last-season records and to warm up the Elo ratings in `data_transform.ipynb`).
- **Split by season, about 70/15/15:** train on 2004-2019, pick settings on 2020-2022 (validation) and test on 2023-2025, which the models never see until the very end.
- **Three models:** logistic regression (the baseline), a random forest and XGBoost. All three use the same features, including the Elo rating built in `data_transform.ipynb`.
- **Compare against two simple baselines:** always pick the home team, and pick the better-ranked team.

**Run `data_exploration.ipynb` and then `data_transform.ipynb` first.**

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay)

## Load the Pre-Game Features

In [ ]:
games = pd.read_csv('datasets/cfb_pregame_features.csv', parse_dates=['date'])

# Home-minus-away pre-game stats (built in data_transform.ipynb)
DIFF_FEATURES = [
    'diff_points_scored',
    'diff_points_allowed',
    'diff_yards_gained',
    'diff_yards_allowed',
    'diff_turnovers_lost',
    'diff_turnovers_forced',
    'diff_third_down_pct',
    'diff_penalty_yards',
    'diff_win_pct',
    'diff_last_season_win_pct',
    'elo_diff',  # Elo rating gap (Step 5 of data_transform.ipynb)
]

# Other things known before kickoff
GAME_FEATURES = [
    'neutral',           # 1 if played at a neutral site
    'conf_game',         # 1 if both teams are in the same conference
    'rank_home_filled',  # AP rank going into the game, unranked = 30
    'rank_away_filled',
]

FEATURES = DIFF_FEATURES + GAME_FEATURES
TARGET = 'home_win'

## Split by Season

Whole seasons go into each part, in order:
- **Train (2004-2019):** every model learns from these games.
- **Validation (2020-2022):** used to pick model settings, like the random forest's tree depth.
- **Test (2023-2025):** only used at the very end, to score each model once.

Splitting by season instead of shuffling means a model never learns from games played after the ones it's tested on. A season can't be cut in half, so the split comes out close to 70/15/15 instead of exactly.

In [ ]:
train = games[(games['season'] >= 2004) & (games['season'] <= 2019)]
validation = games[(games['season'] >= 2020) & (games['season'] <= 2022)]
test = games[(games['season'] >= 2023) & (games['season'] <= 2025)]

X_train = train[FEATURES]
y_train = train[TARGET]
X_val = validation[FEATURES]
y_val = validation[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]

total = len(train) + len(validation) + len(test)  # all 2004-2025 games (2002-2003 aren't used here)
print('Training games (2004-2019):  ', len(train), '=', round(len(train) / total * 100), '%')
print('Validation games (2020-2022):', len(validation), '=', round(len(validation) / total * 100), '%')
print('Test games (2023-2025):      ', len(test), '=', round(len(test) / total * 100), '%')
print('Home team won', round(y_train.mean() * 100, 1), '% of training games and',
      round(y_test.mean() * 100, 1), '% of test games')

## Simple Baselines

A model is only useful if it beats these:
1. **Always pick the home team.**
2. **Pick the better-ranked team.** Lower rank number is better, and unranked teams count as 30. If both teams have the same rank (usually both unranked), pick the home team.

In [ ]:
# Baseline 1: predict 1 (home team wins) for every game
always_home_pred = pd.Series(1, index=test.index)

# Baseline 2: predict 1 if the home team is ranked the same or better than the away team
better_ranked_pred = (test['rank_home_filled'] <= test['rank_away_filled']).astype(int)

# First Model: Logistic Regression

Our baseline model. It's simple, and its coefficients show which way each feature pushes the prediction.

## Train the Logistic Regression

The model is a pipeline of three steps:
1. `SimpleImputer` fills the few blank values (e.g. a team's first game in its first FBS season) with the median from the training data.
2. `StandardScaler` puts every feature on the same scale, so the model treats yards and percentages fairly and we can compare the coefficients later.
3. `LogisticRegression` learns how much each feature pushes the odds of a home win up or down.

In [ ]:
model = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)
model.fit(X_train, y_train)

test_pred = model.predict(X_test)              # 1 = home team wins, 0 = away team wins
test_prob = model.predict_proba(X_test)[:, 1]  # model's chance that the home team wins

print('Training accuracy:', round(accuracy_score(y_train, model.predict(X_train)), 3))
print('Test accuracy:    ', round(accuracy_score(y_test, test_pred), 3))

## Results on the Test Seasons (2023-2025)

The positive class is **home team wins**:
- **Precision:** when we pick the home team, how often it actually wins.
- **Recall:** of the games the home team actually won, how many we picked.
- **ROC-AUC:** how well the predicted probabilities rank home wins above home losses. The simple baselines only give a yes/no pick, not a probability, so they don't get one.

In [ ]:
def score_predictions(name, y_true, y_pred, y_prob=None):
    """Return one row of metrics for a set of predictions."""
    row = {
        'model': name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'roc_auc': float('nan'),
    }
    if y_prob is not None:
        row['roc_auc'] = roc_auc_score(y_true, y_prob)
    return row


results = pd.DataFrame([
    score_predictions('Always pick home team', y_test, always_home_pred),
    score_predictions('Pick better-ranked team', y_test, better_ranked_pred),
    score_predictions('Logistic regression', y_test, test_pred, test_prob),
])
results.round(3)

"Always pick the home team" gets a perfect recall of 1.0 just by never picking the away team. That's why we look at accuracy, F1 and ROC-AUC together instead of any one number.

### Confusion Matrix

Each row adds up to 100%. The top row is games the away team actually won, and the bottom row is games the home team actually won, so the diagonal shows how often the model got each kind of game right. Home teams win more often, so with raw counts the home-win row would look darker just because it has more games.

All three models use the same colors and the same 0-100% scale, so the same shade means the same rate in every chart.

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, test_pred,
                                        display_labels=['Away win', 'Home win'],
                                        normalize='true',             # each row adds up to 100%
                                        values_format='.0%',          # show percentages instead of counts
                                        cmap='Blues',
                                        im_kw={'vmin': 0, 'vmax': 1})  # same 0-100% color scale for every model
plt.title('Logistic regression on 2023-2025 test games')
plt.show()

### ROC Curve

In [ ]:
RocCurveDisplay.from_predictions(y_test, test_prob, name='Logistic regression')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guessing')
plt.title('ROC curve on 2023-2025 test games')
plt.legend()
plt.show()

## Which Features Matter Most?

Because the features were scaled, the coefficients can be compared. A positive bar pushes the prediction toward the home team, a negative bar toward the away team. Features that move together (like points scored and yards gained) end up sharing credit, so read this as a rough guide.

Elo is built from wins and losses, so it overlaps with `diff_win_pct` and `diff_last_season_win_pct`. Elo takes most of their credit, and their bars can even come out negative. That doesn't mean winning hurts. It means Elo already says it better.

In [ ]:
logreg = model.named_steps['logisticregression']
coefficients = pd.Series(logreg.coef_[0], index=FEATURES).sort_values()

coefficients.plot.barh(figsize=(8, 6))
plt.axvline(0, color='gray')
plt.title('Logistic regression coefficients (scaled features)')
plt.xlabel('Coefficient')
plt.show()

## Where Does the Model Miss?

Accuracy by week across the three test seasons. Bowl and playoff games are week 16 in some seasons and week 17 in others, so they're all grouped as week 17. Week 15 is conference championship weekend and week 16 is a single late-season game from 2024, so those two bars swing a lot. The table under the chart shows how many games are behind each bar.

In [ ]:
test_results = test[['week', 'postseason']].copy()
test_results['correct'] = (test_pred == y_test).astype(int)

# Bowl and playoff games are week 16 in some seasons and week 17 in others. Put them all in week 17.
test_results.loc[test_results['postseason'] == 1, 'week'] = 17

by_week = test_results.groupby('week')['correct'].agg(accuracy='mean', games='count')

by_week['accuracy'].plot.bar(figsize=(10, 4), label='Accuracy')
plt.axhline(test_results['correct'].mean(), color='gray', linestyle='--', label='Overall')
plt.ylim(0, 1)
plt.ylabel('Accuracy')
plt.title('Logistic regression accuracy by week (2023-2025)')
plt.legend()
plt.show()

by_week.T.round(2)

# Second Model: Random Forest

Same task, same features and the same test seasons (2023-2025) as the logistic regression above, so the two models can be compared directly. **Run the logistic regression cells first.** This part reuses `games`, `FEATURES`, the train/test split, the two simple baselines and `score_predictions`.

A random forest is a big group of decision trees. Each tree is trained on a random sample of the training games and only looks at a random handful of features at each split, so the trees make different mistakes. The forest averages their votes. Unlike logistic regression, it can pick up interactions between features and effects that aren't a straight line, without us building those by hand.

The experiment below trains 12 forests, so this part takes a minute or so to run.

## Setup

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

## The Pipeline

Two steps this time:
1. `SimpleImputer` fills blank values with the training median, same as before.
2. `RandomForestClassifier` builds the trees.

There's no `StandardScaler`. A tree splits on one feature at a time (for example, "is `diff_win_pct` above 0.2?"), so changing a feature's scale doesn't change where it splits.

The function below builds the pipeline for a given pair of tree settings, so the experiment can try several:
- **`max_depth`:** how many splits deep each tree can go. `None` means no limit, so a tree keeps splitting until every leaf holds games with the same result.
- **`min_samples_leaf`:** the fewest training games allowed in a leaf. Bigger values force broader, smoother rules.

Both settings control how closely each tree can memorize the training games.

In [ ]:
def make_random_forest(max_depth, min_samples_leaf):
    """Build an imputer + random forest pipeline with the given tree settings."""
    return make_pipeline(
        SimpleImputer(strategy='median'),
        RandomForestClassifier(
            n_estimators=300,  # number of trees
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=42,   # same trees every run
            n_jobs=-1,         # use all CPU cores
        ),
    )

## Experiment: Picking the Tree Settings

We can't pick the settings by checking them on the test seasons, or they stop being a fair test. That's what the **validation seasons (2020-2022)** are for: each setting is trained on the training seasons (2004-2019) and scored on the validation seasons, and the best one wins. The test seasons are only used once, at the very end.

The best setting is picked by validation ROC-AUC instead of accuracy. Accuracy can move a point or two from a few close games going the other way, while ROC-AUC uses the full predicted probabilities, so it's steadier.

In [ ]:
experiment_rows = []
best_auc = 0

for max_depth in [4, 8, 12, None]:
    for min_samples_leaf in [1, 20, 50]:
        forest = make_random_forest(max_depth, min_samples_leaf)
        forest.fit(X_train, y_train)

        val_prob = forest.predict_proba(X_val)[:, 1]
        val_auc = roc_auc_score(y_val, val_prob)

        experiment_rows.append({
            'max_depth': str(max_depth),  # str() so None shows up in the table
            'min_samples_leaf': min_samples_leaf,
            'train_accuracy': accuracy_score(y_train, forest.predict(X_train)),
            'val_accuracy': accuracy_score(y_val, forest.predict(X_val)),
            'val_roc_auc': val_auc,
        })

        # Remember the best settings so far
        if val_auc > best_auc:
            best_auc = val_auc
            best_max_depth = max_depth
            best_min_samples_leaf = min_samples_leaf

experiment = pd.DataFrame(experiment_rows)

print('Best settings: max_depth =', best_max_depth, '| min_samples_leaf =', best_min_samples_leaf)
experiment.sort_values('val_roc_auc', ascending=False).round(3)

For reference, here's how the logistic regression from the first part does on the validation seasons. If the best forest isn't clearly ahead of this on validation, don't expect it to be clearly ahead on the test seasons either.

In [ ]:
# The logistic regression was already trained on the training seasons, so just score it on validation
print('Logistic regression validation accuracy:', round(accuracy_score(y_val, model.predict(X_val)), 3))
print('Logistic regression validation ROC-AUC: ',
      round(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]), 3))

### Do Deeper Trees Overfit?

Training vs. validation accuracy as the trees get deeper (with `min_samples_leaf = 1`, the loosest setting). A single decision tree with no depth limit would memorize the training games and do badly on new ones. A forest averages hundreds of different trees, so the memorizing mostly cancels out. Watch whether the validation line drops as the training line climbs.

In [ ]:
loosest_leaf = experiment[experiment['min_samples_leaf'] == 1]

plt.plot(loosest_leaf['max_depth'], loosest_leaf['train_accuracy'], marker='o', label='Training (2004-2019)')
plt.plot(loosest_leaf['max_depth'], loosest_leaf['val_accuracy'], marker='o', label='Validation (2020-2022)')
plt.ylim(0.5, 1.02)
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Random forest accuracy by tree depth (min_samples_leaf = 1)')
plt.legend()
plt.show()

## Train the Final Random Forest

Build the forest again with the best settings, train it on the training seasons (the same ones the logistic regression used) and predict the test seasons.

If the best settings have no depth limit, training accuracy will be close to 1.0. That's normal for a random forest: each tree memorizes the games it was trained on, and the averaging is what keeps the forest from overfitting. The test score is the number that matters.

In [ ]:
rf_model = make_random_forest(best_max_depth, best_min_samples_leaf)
rf_model.fit(X_train, y_train)

rf_test_pred = rf_model.predict(X_test)              # 1 = home team wins, 0 = away team wins
rf_test_prob = rf_model.predict_proba(X_test)[:, 1]  # forest's chance that the home team wins

print('Training accuracy:', round(accuracy_score(y_train, rf_model.predict(X_train)), 3))
print('Test accuracy:    ', round(accuracy_score(y_test, rf_test_pred), 3))

## Results on the Test Seasons (2023-2025)

Same metrics as before, with the random forest added. `test_pred` and `test_prob` are still the logistic regression's predictions from the first part.

In [ ]:
all_results = pd.DataFrame([
    score_predictions('Always pick home team', y_test, always_home_pred),
    score_predictions('Pick better-ranked team', y_test, better_ranked_pred),
    score_predictions('Logistic regression', y_test, test_pred, test_prob),
    score_predictions('Random forest', y_test, rf_test_pred, rf_test_prob),
])
all_results.round(3)

### Confusion Matrix

Same layout and color scale as the logistic regression's: each row adds up to 100%.

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, rf_test_pred,
                                        display_labels=['Away win', 'Home win'],
                                        normalize='true',             # each row adds up to 100%
                                        values_format='.0%',          # show percentages instead of counts
                                        cmap='Blues',
                                        im_kw={'vmin': 0, 'vmax': 1})  # same 0-100% color scale for every model
plt.title('Random forest on 2023-2025 test games')
plt.show()

### ROC Curves

Both models on the same chart. The closer a curve gets to the top-left corner, the better that model separates home wins from home losses.

In [ ]:
fig, ax = plt.subplots()
RocCurveDisplay.from_predictions(y_test, test_prob, name='Logistic regression', ax=ax)
RocCurveDisplay.from_predictions(y_test, rf_test_prob, name='Random forest', ax=ax)
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guessing')
ax.set_title('ROC curves on 2023-2025 test games')
ax.legend()
plt.show()

## Which Features Matter Most?

A random forest has no coefficients, so this uses **permutation importance** instead. For each feature, its values in the test games are shuffled randomly (10 times), and we measure how much the model's ROC-AUC drops. A big drop means the model leans on that feature. A drop near zero means the model barely uses it.

Running it on both models makes the comparison fair, since the same test is applied to each. Unlike the coefficients in the first part, this doesn't say which direction a feature pushes the prediction, only how much the model relies on it.

The forest also has a built-in `feature_importances_`, but it tends to overrate features with lots of different values (like yards) and underrate yes/no features (like `neutral`), so permutation importance is the more trustworthy view.

In [ ]:
def get_permutation_importance(fitted_model):
    """Average drop in test-season ROC-AUC when each feature is shuffled."""
    result = permutation_importance(fitted_model, X_test, y_test,
                                    scoring='roc_auc', n_repeats=10, random_state=42)
    return pd.Series(result.importances_mean, index=FEATURES)


importance = pd.DataFrame({
    'Logistic regression': get_permutation_importance(model),
    'Random forest': get_permutation_importance(rf_model),
}).sort_values('Random forest')

importance.plot.barh(figsize=(8, 7))
plt.axvline(0, color='gray')
plt.title('Permutation importance on 2023-2025 test games')
plt.xlabel('Drop in ROC-AUC when the feature is shuffled')
plt.show()

## Where Does Each Model Miss?

Accuracy by week of the test seasons for both models, with bowl and playoff games grouped in week 17 like before. The number of games in each week is in the table under the logistic regression's chart above. Weeks 15 and 16 only have a handful of games, so their bars swing a lot.

In [ ]:
weekly_correct = pd.DataFrame({
    'week': test_results['week'],  # bowl games grouped in week 17 above
    'Logistic regression': (test_pred == y_test).astype(int),
    'Random forest': (rf_test_pred == y_test).astype(int),
})
by_week_both = weekly_correct.groupby('week').mean()

by_week_both.plot.bar(figsize=(10, 4))
plt.ylim(0, 1)
plt.ylabel('Accuracy')
plt.title('Accuracy by week (2023-2025)')
plt.legend(loc='upper center', ncol=2)
plt.show()

by_week_both.T.round(2)

# Third Model: XGBoost

Same task, same features (Elo included) and the same test seasons (2023-2025) as the two models above. **Run the logistic regression and random forest cells first.** This part reuses `FEATURES`, the train/test split, `score_predictions` and both models' predictions.

XGBoost is also a group of decision trees, but it builds them **one at a time**. Each new tree focuses on the games the trees before it got wrong, and the final prediction adds up all the trees. A random forest builds its trees separately and averages their votes.

## Setup

In [ ]:
from xgboost import XGBClassifier

## Train XGBoost

We keep the settings simple:
- **`n_estimators=200`:** build 200 trees.
- **`max_depth=3`:** keep each tree small (at most 3 splits deep) so it can't memorize the training games.
- **`learning_rate=0.05`:** each tree only nudges the prediction a little, so no single tree takes over.

These are common starting values for XGBoost. We tried a few other combinations on the validation seasons and none did clearly better, so we kept these. The test seasons weren't used to pick them.

There's no imputer or scaler. XGBoost handles blank values by itself, and like the random forest, trees don't care about scale.

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=200,     # build 200 small trees, each one fixing some of the last one's mistakes
    max_depth=3,          # keep each tree small so it can't memorize the training games
    learning_rate=0.05,   # each tree only nudges the prediction a little
    random_state=42,      # same result every run
)
xgb_model.fit(X_train, y_train)

xgb_test_pred = xgb_model.predict(X_test)              # 1 = home team wins, 0 = away team wins
xgb_test_prob = xgb_model.predict_proba(X_test)[:, 1]  # XGBoost's chance that the home team wins

print('Training accuracy:', round(accuracy_score(y_train, xgb_model.predict(X_train)), 3))
print('Test accuracy:    ', round(accuracy_score(y_test, xgb_test_pred), 3))

## Results on the Test Seasons (2023-2025)

All three models and both baselines side by side. Even with about 2,400 test games, accuracy can move about a point from a few close games going the other way, so models within a point of each other are basically tied.

In [ ]:
all_results = pd.DataFrame([
    score_predictions('Always pick home team', y_test, always_home_pred),
    score_predictions('Pick better-ranked team', y_test, better_ranked_pred),
    score_predictions('Logistic regression', y_test, test_pred, test_prob),
    score_predictions('Random forest', y_test, rf_test_pred, rf_test_prob),
    score_predictions('XGBoost', y_test, xgb_test_pred, xgb_test_prob),
])
all_results.round(3)

### Confusion Matrix

Same layout and color scale as the logistic regression's: each row adds up to 100%.

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, xgb_test_pred,
                                        display_labels=['Away win', 'Home win'],
                                        normalize='true',             # each row adds up to 100%
                                        values_format='.0%',          # show percentages instead of counts
                                        cmap='Blues',
                                        im_kw={'vmin': 0, 'vmax': 1})  # same 0-100% color scale for every model
plt.title('XGBoost on 2023-2025 test games')
plt.show()

### ROC Curves

All three models on the same chart. The closer a curve gets to the top-left corner, the better that model separates home wins from home losses.

In [ ]:
fig, ax = plt.subplots()
RocCurveDisplay.from_predictions(y_test, test_prob, name='Logistic regression', ax=ax)
RocCurveDisplay.from_predictions(y_test, rf_test_prob, name='Random forest', ax=ax)
RocCurveDisplay.from_predictions(y_test, xgb_test_prob, name='XGBoost', ax=ax)
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guessing')
ax.set_title('ROC curves on 2023-2025 test games')
ax.legend()
plt.show()

# Extra Plots for the Presentation

Seven plots of the three trained models on the 2023-2025 test games. **Run every cell above first.** Nothing is retrained here: every plot uses the train/validation/test split, the results table, and the predictions and probabilities the models already made. Each plot is also saved as a PNG in the `figures` folder so it can go straight into the slides.

In [ ]:
import os
import numpy as np
from matplotlib.ticker import PercentFormatter

os.makedirs('figures', exist_ok=True)  # folder for the PNGs

plt.rcParams['font.size'] = 12            # bigger text so it's readable on a slide
plt.rcParams['axes.spines.top'] = False   # drop the top and right edges of each plot
plt.rcParams['axes.spines.right'] = False

# Each model keeps the same color in every plot below
COLORS = {
    'Logistic regression': '#2a78d6',
    'Random forest': '#eb6834',
    'XGBoost': '#1baf7a',
    'Baseline': '#a3a29c',
}
MODEL_NAMES = ['Logistic regression', 'Random forest', 'XGBoost']
HOME_COLOR = '#6250d6'  # home win
AWAY_COLOR = '#eda100'  # away win

### Plot 4: Games in Each Season, by Split

How many games each season has and which part of the split it went to. The models only ever learn from the gray seasons and are scored on the dark ones, so every test game is in the future compared to the training games.

In [ ]:
splits = [
    ('Training', train, '#c3c2bc'),
    ('Validation', validation, '#7d7c77'),
    ('Test', test, '#2e2d2a'),
]

plt.figure(figsize=(10, 4.5))
for name, part, color in splits:
    games_per_season = part.groupby('season').size()
    first, last = part['season'].min(), part['season'].max()
    plt.bar(games_per_season.index, games_per_season, color=color,
            label=f'{name} {first}-{last} ({len(part):,})')

plt.xticks(range(2004, 2026, 3))
plt.ylim(0, 1000)
plt.xlabel('Season')
plt.ylabel('FBS games')
plt.title('Games per season and how they were split')
plt.legend(loc='upper center', ncol=3)
plt.tight_layout()
plt.savefig('figures/plot04_games_by_split.png', dpi=200, bbox_inches='tight')
plt.show()

### Plot 5: Test Accuracy, Models vs. Baselines

All five ways of picking games from `all_results`, on the same 2023-2025 test games. The bars start at 0 so the gaps aren't exaggerated.

In [ ]:
names = ['Always pick home team', 'Pick better-ranked team', 'Logistic regression', 'Random forest', 'XGBoost']
accuracy = all_results.set_index('model').loc[names, 'accuracy']
bar_colors = [COLORS['Baseline'], COLORS['Baseline'],
              COLORS['Logistic regression'], COLORS['Random forest'], COLORS['XGBoost']]

plt.figure(figsize=(9, 4.5))
bars = plt.barh(names, accuracy, color=bar_colors)
plt.bar_label(bars, labels=[f'{a:.1%}' for a in accuracy], padding=4)
plt.gca().invert_yaxis()  # keep the list in the same order as names, top to bottom
plt.xlim(0, 1)
plt.gca().xaxis.set_major_formatter(PercentFormatter(1))
plt.xlabel('Accuracy on 2023-2025 test games')
plt.title('Test accuracy: models vs. baselines')
plt.tight_layout()
plt.savefig('figures/plot05_accuracy_vs_baselines.png', dpi=200, bbox_inches='tight')
plt.show()

### Plot 6: Accuracy, F1 and ROC-AUC Side by Side

The three numbers we report for each model, straight from `all_results`.

In [ ]:
metrics = all_results.set_index('model').loc[MODEL_NAMES, ['accuracy', 'f1', 'roc_auc']]
metrics.columns = ['Accuracy', 'F1', 'ROC-AUC']

ax = metrics.T.plot.bar(figsize=(9, 5), width=0.75, color=[COLORS[name] for name in MODEL_NAMES])
for bars in ax.containers:
    ax.bar_label(bars, labels=[f'{v:.3f}' for v in bars.datavalues], padding=3, fontsize=9)
plt.xticks(rotation=0)
plt.ylim(0, 1.1)
plt.ylabel('Score on 2023-2025 test games')
plt.title('Accuracy, F1 and ROC-AUC for each model')
plt.legend(loc='upper center', ncol=3)
plt.tight_layout()
plt.savefig('figures/plot06_metrics_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

### Plot 7: Home Wins vs. Away Wins Called Right

For each model: out of the test games the home team actually won, how many did it pick right? Same for the games the away team won. All three models lean toward the home team.

In [ ]:
home_won = y_test.values == 1
away_won = y_test.values == 0
model_preds = [test_pred, rf_test_pred, xgb_test_pred]

calls = pd.DataFrame({
    'Home wins called right': [(pred[home_won] == 1).mean() for pred in model_preds],
    'Away wins called right': [(pred[away_won] == 0).mean() for pred in model_preds],
}, index=MODEL_NAMES)

ax = calls.plot.bar(figsize=(9, 5), width=0.7, color=[HOME_COLOR, AWAY_COLOR])
for bars in ax.containers:
    ax.bar_label(bars, labels=[f'{v:.0%}' for v in bars.datavalues], padding=3)
plt.xticks(rotation=0)
plt.ylim(0, 1.1)
plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
plt.ylabel('Share called right')
plt.title('Home wins vs. away wins called right (2023-2025)')
plt.legend(loc='upper center', ncol=2)
plt.tight_layout()
plt.savefig('figures/plot07_home_vs_away_calls.png', dpi=200, bbox_inches='tight')
plt.show()

calls.round(3)

### Plot 8: Accuracy by How Confident the Model Was

For each test game, take the logistic regression's chance for the team it picked (50% = coin flip, 100% = certain). Group the games by that confidence and check how often the pick was right.

In [ ]:
# Chance the model gave to the team it picked
confidence = np.maximum(test_prob, 1 - test_prob)
confidence_group = pd.cut(confidence, bins=[0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
                          labels=['50-60%', '60-70%', '70-80%', '80-90%', '90-100%'],
                          include_lowest=True)

by_confidence = pd.DataFrame({'group': confidence_group, 'correct': test_pred == y_test.values})
by_confidence = by_confidence.groupby('group', observed=True)['correct'].agg(accuracy='mean', games='count')

plt.figure(figsize=(8, 5))
bars = plt.bar(by_confidence.index, by_confidence['accuracy'], color=COLORS['Logistic regression'])
bar_labels = [f'{a:.0%}\n{g} games' for a, g in zip(by_confidence['accuracy'], by_confidence['games'])]
plt.bar_label(bars, labels=bar_labels, padding=4)
plt.axhline(0.5, color='gray', linestyle='--', label='Coin flip')
plt.ylim(0, 1.15)
plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
plt.xlabel("Model's chance for the team it picked")
plt.ylabel('Picks that were right')
plt.title('Logistic regression accuracy by confidence (2023-2025)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.savefig('figures/plot08_accuracy_by_confidence.png', dpi=200, bbox_inches='tight')
plt.show()

by_confidence.round(3)

### Plot 9: Accuracy by How Close the Matchup Was

Group the test games by the Elo gap between the teams before kickoff (the `elo_diff` column, ignoring which side was ahead). Close matchups are hard for every model; mismatches are easy.

In [ ]:
gap_size = pd.cut(test['elo_diff'].abs(), bins=[0, 100, 200, 300, 2000],
                  labels=['0-100', '100-200', '200-300', '300+'], right=False)

correct_by_gap = pd.DataFrame({
    'gap': gap_size,
    'Logistic regression': (test_pred == y_test).astype(int),
    'Random forest': (rf_test_pred == y_test).astype(int),
    'XGBoost': (xgb_test_pred == y_test).astype(int),
})
accuracy_by_gap = correct_by_gap.groupby('gap', observed=True)[MODEL_NAMES].mean()
games_by_gap = correct_by_gap.groupby('gap', observed=True).size()

ax = accuracy_by_gap.plot.bar(figsize=(9, 5), width=0.75, color=[COLORS[name] for name in MODEL_NAMES])
for bars in ax.containers:
    ax.bar_label(bars, labels=[f'{v:.0%}' for v in bars.datavalues], padding=3, fontsize=10)
ax.set_xticks(range(len(games_by_gap)),
              [f'{gap}\n({n} games)' for gap, n in zip(games_by_gap.index, games_by_gap)], rotation=0)
plt.axhline(0.5, color='gray', linestyle='--', label='Coin flip')
plt.ylim(0, 1.15)
plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
plt.xlabel('Elo gap before kickoff (either direction)')
plt.ylabel('Accuracy')
plt.title('Accuracy by how close the matchup was (2023-2025)')
plt.legend(loc='upper left', ncol=2)
plt.tight_layout()
plt.savefig('figures/plot09_accuracy_by_elo_gap.png', dpi=200, bbox_inches='tight')
plt.show()

accuracy_by_gap.round(3)

### Plot 10: XGBoost's Built-In Feature Importance

A trained XGBoost model keeps its own importance score for each feature (`feature_importances_`): how much that feature improved the trees' splits, as a share of the total. The 10 most important features are shown, with plain-English names.

In [ ]:
NICE_NAMES = {
    'elo_diff': 'Elo gap',
    'diff_win_pct': 'Win % gap',
    'diff_last_season_win_pct': "Last season's win % gap",
    'diff_points_scored': 'Points scored gap',
    'diff_points_allowed': 'Points allowed gap',
    'diff_yards_gained': 'Yards gained gap',
    'diff_yards_allowed': 'Yards allowed gap',
    'diff_turnovers_lost': 'Turnovers lost gap',
    'diff_turnovers_forced': 'Turnovers forced gap',
    'diff_third_down_pct': '3rd-down % gap',
    'diff_penalty_yards': 'Penalty yards gap',
    'neutral': 'Neutral site',
    'conf_game': 'Conference game',
    'rank_home_filled': 'Home AP rank',
    'rank_away_filled': 'Away AP rank',
}

xgb_importance = pd.Series(xgb_model.feature_importances_, index=FEATURES).rename(index=NICE_NAMES)
top_xgb = xgb_importance.sort_values().tail(10)  # smallest first, so the biggest ends up on top

plt.figure(figsize=(8, 5))
bars = plt.barh(top_xgb.index, top_xgb, color=COLORS['XGBoost'])
plt.bar_label(bars, labels=[f'{v:.0%}' for v in top_xgb], padding=4)
plt.xlim(0, top_xgb.max() * 1.15)
plt.gca().xaxis.set_major_formatter(PercentFormatter(1))
plt.xlabel("Share of XGBoost's total importance")
plt.title("XGBoost's built-in feature importance (top 10)")
plt.tight_layout()
plt.savefig('figures/plot10_xgboost_importance.png', dpi=200, bbox_inches='tight')
plt.show()